## 6.S083 / 18.S190
# PSET 1 Solution – Data Analysis


In [ ]:
# import Pkg

# Pkg.add("CSV")
# Pkg.add("HTTP")
# Pkg.add("Plots")
# Pkg.add("Interact")
# Pkg.add("PlotlyJS")
# Pkg.add("DataFrames")
# Pkg.add("DataStructures")



## Exercise 1: Visualizing the data over time

------


In [ ]:
using HTTP

case_csv = HTTP.get(
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
).body;


### 1. Extract the data and country names from the CSV file as we did in class

Call the country names `all_countries`


In [ ]:
using CSV
using DataFrames

case_data = CSV.read(case_csv);

rename!(
  case_data,
  "Province/State" => "province",
  "Country/Region" => "country"
)

all_countries = unique(case_data.country);


In [ ]:
using Plots
using DataStructures

cur_counter = counter(case_data.country)

cur_counter = filter(
  cur_tuple -> cur_tuple[2] > 1, cur_counter
)

cur_counter = OrderedDict(
  reverse(sort(collect(cur_counter), by=x->x[2]))
)

bar(cur_counter, label="", title="Province Counts", ylabel="Province Count", xlabel="Country")



------

### 2. Make a `Vector` called `countries` with a subset of countries that we wish to plot 

Say China, Japan, South Korea, US, United Kingdom, France, Germany. Be careful to check how they are written in the data set


In [ ]:
# note that "South Korea" => "Korea, South"

countries = sort([
  "China", "Japan", "Korea, South", "US", "United Kingdom", "France", "Germany"
])

country_data = case_data[map(cur_country -> cur_country in countries, case_data.country), :];


In [ ]:
# note that France and UK include their colonies

scatter(
  country_data.Long, country_data.Lat, group=country_data.country, 
  legend=:bottomleft, xlims=(-180,+180), ylim=(-60,+60),
  xlabel="Longitude", ylabel="Latitude", title="Province Locations"
)


-----

### 3. Define a variable `num_days` by extracting the number of days of data from the dataframe.


In [ ]:
using Dates

column_names = names(country_data)
day_columns = column_names[count.("/", map(string, column_names)) .== 2]

cur_dates = Date.(map(string, day_columns), DateFormat("m/d/y")) + Dates.Year(2000);
@show num_days = length(cur_dates);


-----

### 4. We need to accumulate the data for those places that are split up into territories. 

+ Make a zero vector of the correct length for each country, e.g. using the function `zeros`.

+ Loop through all the countries and add the corresponding data to that country's data.

+ You may use a dictionary (`Dict`), or a matrix, or a `Vector` containing `Vector`s, or a new `DataFrame` to store the data.


In [ ]:
# data frame approach

aggregate_df = aggregate(
  groupby(
    country_data[:, [:country, day_columns...]], 
    :country
  ), 
  sum
)

# dict of lists approach

cur_dict = OrderedDict()
for country in countries
  cur_dict[country] = sum.(
    eachcol(country_data[country_data.country .== country, day_columns])
  )
end

# matrix approach

cur_matrix = zeros(Int, length(countries), num_days)
for (i, country) in enumerate(countries)
  sub_data = country_data[country_data.country .== country, day_columns]
  for cur_row in eachrow(sub_data)
    for j in 1:num_days
      cur_matrix[i,j] += cur_row[j]
    end
  end
end


-----

### 5. Plot all countries' data on the same graph by using a `for` loop.


Then run the loop, adding in the data using `plot!`

Finally, display the plot by evaluating the plot object `p`. (Just type its name and evaluate.)


In [ ]:
p = plot(legend=:topleft, title="Cumulative Cases")

for (country, cases) in cur_dict
  plot!(cur_dates, cases, label=country)
end

p


-----

### 6. Now use a `log` scale on the $y$ axis. 

In order to do so you will:

+ need to convert the vectors to contain `Float64` 
+ replace any 0 values by `NaN` ("not a number") so that `Plots.jl` ignores those values


In [ ]:
p = plot(legend=:bottomright, title="Log Cumulative Cases")

for (country, cases) in cur_dict
  plot!(cur_dates, replace(cases, 0 => NaN), label=country, yscale=:log10)
end

p


*Q: Is there exponential growth?*

***A: Yes there is exponential growth. After reaching 10 cases, most curves are approximately linear (or sigmoidal for South Korea)***

<br>

-----

### 7. Turn this into an interactive visualization by adding a slider corresponding to the current day

Varying the day between 1 and the total number of days for which you have data.

+ You should draw only the data up to that particular day. 
+ As you move the slider the plot should update. 
+ Fix the horizontal axis across slider changes

In [ ]:
function custom_plot(yscale, cur_day)
  p = plot(legend=:bottomright, title="Cumulative Cases")

  for (country, cases) in cur_dict
    plot!(cur_dates[1:cur_day], replace(cases, 0 => NaN)[1:cur_day], label=country, markershape=:circle)
  end
  
  xlims!(map(Dates.value, (cur_dates[1], cur_dates[end])))
  ylims!(1, 2*maximum(Base.Iterators.flatten(values(cur_dict))))
  
  plot!(yscale=yscale)
end


In [ ]:
using Interact

@manipulate for yscale=[:identity, :log10], cur_day in 1:num_days
  custom_plot(yscale, cur_day)
end


----

## Exercise 2: Visualizing changes

Now let's try to reproduce the essence of the nice visualization from https://aatishb.com/covidtrends, which is a less usual point of view. 

+ Again the slider will represent a day during the epidemic.
+ The horizontal axis will show the *total* confirmed cases until the given day
+ While the vertical axis will show the *change* in confirmed cases during the past week (7 days).


In [ ]:
function make_widget(cur_day, widget_dict=cur_dict, add_annotation=false)
  p = plot(
    legend=:topleft, xlabel="Total Cases", ylabel="Recent Cases", 
    title="Trajectory of COVID-19 Confirmed Cases (" * string(day_columns[cur_day]) * ")",
    xscale=:log10, yscale=:log10
  )
  
  lims_dict = Dict(
    "min_x" => +Inf, "max_x" => -Inf,
    "min_y" => +Inf, "max_y" => -Inf,
  )
  
  for (cur_index, (country, cases)) in enumerate(deepcopy(widget_dict))
    cases = Array{AbstractFloat}(cases)
    
    recent_cases = make_recent_cases(cases)
    update_limits!(lims_dict, cases, recent_cases)

    cases = cases[1:cur_day]
    recent_cases =  recent_cases[1:cur_day]
    
    remove_small_values!(cases, recent_cases)
    isempty(cases) && continue

    plot!(cases, recent_cases, label=country, color=cur_index)
    
    add_annotation && add_annotations!(
      country, cur_index, cases[end], recent_cases[end]
    )
  end
  
  xlims!(lims_dict["min_x"]/2, lims_dict["max_x"]*2)
  ylims!(lims_dict["min_y"]/2, lims_dict["max_y"]*2)
end


In [ ]:
function make_recent_cases(cases, delay=7) 
  return cases - [ zeros(Int, delay)..., cases[1:end-delay]... ]
end

function update_limits!(lims_dict, cases, recent_cases)
  for (tmp_key, tmp_cases) in Dict( :x => cases, :y => recent_cases )
    lims_dict["max_$(tmp_key)"] = max(lims_dict["max_$(tmp_key)"], maximum(tmp_cases))

    filter_cases = filter(!iszero, tmp_cases)
    if !isempty(filter_cases)
      lims_dict["min_$(tmp_key)"] = min(lims_dict["min_$(tmp_key)"], minimum(filter_cases)) 
    end
  end
end

function remove_small_values!(cases, recent_cases, min_cases=8, min_recent_cases=4)
  del_indices = []
  
  for (cur_cases, cur_threshold) in (cases => min_cases, recent_cases => min_recent_cases)
    append!(del_indices, findall(cur_case -> cur_case <= cur_threshold, cur_cases))
  end
  
  unique!(del_indices)
  sort!(del_indices)

  deleteat!(cases, del_indices)
  deleteat!(recent_cases, del_indices)
end


In [ ]:
@manipulate for cur_day in 1:num_days
  make_widget(cur_day)
end


----

### 1. Make a data set `total_cases`

Representing for each day the *total* number of confirmed cases during the whole epidemic up until that point


In [ ]:
total_cases = sum.(eachcol(case_data[:, day_columns]));
# total_cases = sum.(eachcol(country_data[:, day_columns]));


----

### 2. Make a set of data `new_cases` which is the total number of cases only during the past 7 days.


In [ ]:
new_cases = total_cases - [ zeros(Int, 7)..., total_cases[1:end-7]... ];
@assert make_recent_cases(total_cases) == new_cases

@assert true
# @assert false    # raises AssertionError


----

### 3. Make the visualization using a slider representing days as in the previous exercise

Plotting the total number of cases on the $x$ axis and new cases on the $y$ axis.

You need to take care that each vector being plotted has the same length.


In [ ]:
all_dict = OrderedDict([
  ("All" => total_cases), 
  cur_dict...
])

@manipulate for cur_day in 1:num_days
  make_widget(cur_day, all_dict)
end


----

### 4. Add a dot for each countries current position and annotate


In [ ]:
# already used inside make_widget function

function add_annotations!(country, cur_index, cases_end, recent_cases_end)
  scatter!([cases_end], [recent_cases_end], label="", color=cur_index, markersize=2.5)
  annotate!(cases_end, recent_cases_end, text(country, 9, :grey, :left))
end


In [ ]:
@manipulate for cur_day in 1:num_days
  add_annotations = true
  
  make_widget(cur_day, all_dict, add_annotations)
end


----

### 5. Make a GIF version of the plot from before



In [ ]:
animation = @animate for cur_day in 1:num_days
  make_widget(cur_day, [("All" => total_cases), cur_dict...], true)
end

gif(animation, fps = 6)


----

## Extra Credit: Web Dashboard

In [ ]:
using JSON

macro jsonplots(args...)
    n = length(args)
    @assert n == 2
  
    expr = args[n]
    if expr.head != :for
        error("@manipulate syntax is @manipulate for ",
              " [<variable>=<domain>,]... <expression> end")
    end
  
    cur_block = expr.args[2]
  
    if expr.args[1].head == :block
        cur_bindings = expr.args[1].args
    else
        cur_bindings = [expr.args[1]]
    end
    
    cur_ranges = []
  
    for cur_binding in cur_bindings
      @assert cur_binding.head == :(=)
      cur_symbol, cur_range = cur_binding.args
    
      isa(cur_range, Expr) && ( cur_range = eval(cur_range) )
      isa(cur_range, Bool) && ( cur_range = [cur_range, !cur_range] )
    
      push!(cur_ranges, (string(cur_symbol), [collect(cur_range)...]))
    end
  
    file_name = args[1]
    endswith(file_name, ".json") || ( file_name *= ".json" )
  
    json_dict = OrderedDict(
      "ranges" => cur_ranges, 
      "plots" => compileTree(cur_block, cur_ranges)
    )
  
    open(file_name,"w") do cur_file
        JSON.print(cur_file, json_dict)
    end
  
    return :($(json_dict))
end

function compileTree(cur_block, cur_ranges, cur_dict=OrderedDict())
  if isempty(cur_ranges) 
    cur_plot = eval(cur_block)
    display(cur_plot)
    
    cur_json = Plots.plotlyjs_syncplot(cur_plot).plot
    return cur_json
  end
  
  tmp_ranges = deepcopy(cur_ranges)
  cur_symbol, cur_range = popfirst!(tmp_ranges)
  
  sub_dict = OrderedDict()
  
  for cur_value in cur_range
    tmp_block = deepcopy(cur_block)
    
    if isa(cur_value, Symbol)
      work_block = quote
        $(Symbol(cur_symbol)) = Symbol($(cur_value))
      end
    else
      work_block = :( $(Symbol(cur_symbol)) = $(cur_value) )
    end

    insert!(tmp_block.args, 2, work_block)
    
    sub_dict[cur_value] = compileTree(tmp_block, tmp_ranges, cur_dict)
  end
  
  return sub_dict
end


In [ ]:
plotlyjs()

@jsonplots "../public/_includes/time-series.json" for yscale=[:identity, :log10], cur_day in 1:num_days
  custom_plot(yscale, cur_day)
end

@jsonplots "../public/_includes/covid-trajectory.json" for cur_day in 1:num_days
  make_widget(cur_day, all_dict, true)
end

gr()
